### Non-Data Descriptor — Lazy / Cached Property

In [ ]:
class CachedProperty:
    """
    Non-data descriptor — computes once, stores in instance __dict__.
    Because it's non-data (no __set__), the instance __dict__ shadows it
    on subsequent access → effectively cached for free.

    Python 3.8+ has functools.cached_property — this shows how it works.
    """
    def __init__(self, func):
        self.func    = func
        self.name    = None
        self.__doc__ = func.__doc__

    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        # Compute and store directly in instance __dict__
        value = self.func(obj)
        obj.__dict__[self.name] = value   # ← shadows this descriptor on next access
        return value


class Order:
    def __init__(self, items: list[dict]):
        self.items = items

    @CachedProperty
    def total(self) -> float:
        """Expensive calculation — runs only once per instance."""
        print("Computing total...")
        return sum(item["price"] * item["qty"] for item in self.items)


o = Order([{"price": 9.99, "qty": 2}, {"price": 4.99, "qty": 1}])
print(o.total)   # "Computing total..." → 24.97
print(o.total)   # no print — served from o.__dict__["total"]
print(o.total)   # no print — still cached